# Feature classifier A – all training data, classes balanced with weights

This notebook trains a simple classifier (logistic regression) that tells bona fide from
spoofed recordings using only the four features of notebook 09: duration, silence at the start
and at the end, loudness and bandwidth.

**Method A:** all recordings of ASVspoof 2019 LA train (2 580 bona fide, 22 800 spoof). Each
bona fide recording counts more, so both classes weigh the same in training. Notebook 10b does
the same with an equal number of recordings per class.

| | data |
|---|---|
| training | ASVspoof 2019 LA train |
| test | ASVspoof 2019 LA eval; ASVspoof 2021 LA and DF (progress, eval, hidden); In-the-Wild |

**How to use it:** run notebook 09 first, then the cells from top to bottom.

**Output files**

- `results/classifier/10a_feature_classifier_all_data_class_weights_output/` (in git):
  `eer_table.csv`, `score_medians.csv`, `coefficients.csv`
- `outputs/10a_feature_classifier_all_data_class_weights_output/` (not in git):
  `scores/scores_<dataset>_<subset>.csv`, `model.joblib`

**Steps**

1. Setup
2. Feature tables
3. Training
4. EER of the test sets
5. Scores per codec
6. Weights of the features
7. Saving the results

### Step 1 – Setup

**What it does:** Sets the paths and lists the training set and the test sets, with a check
that the feature table of each one exists.

**Why:** All the next cells use these paths, and a missing file is visible at once.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display
from sklearn.utils.class_weight import compute_class_weight

from xai_audio_deepfake_shortcuts_audit.classifier.add_scores import add_scores
from xai_audio_deepfake_shortcuts_audit.classifier.config import (
    BREAKDOWNS,
    CODEC_REFERENCES,
    FEATURE_COLUMNS,
    SUFFIX,
    TEST,
    TRAIN,
)
from xai_audio_deepfake_shortcuts_audit.classifier.eer_by_group import eer_by_group
from xai_audio_deepfake_shortcuts_audit.classifier.feature_weights import feature_weights
from xai_audio_deepfake_shortcuts_audit.classifier.score_medians import score_medians
from xai_audio_deepfake_shortcuts_audit.classifier.select_features import select_features
from xai_audio_deepfake_shortcuts_audit.classifier.train_classifier import train_classifier
from xai_audio_deepfake_shortcuts_audit.features.read_feature_table import read_feature_table
from xai_audio_deepfake_shortcuts_audit.utils.find_repo_root import find_repo_root

REPO_ROOT = find_repo_root(Path.cwd())
FEATURES_DIR = REPO_ROOT / "outputs" / "09_feature_extraction_output" / "features"
NOTEBOOK_NAME = "10a_feature_classifier_all_data_class_weights"
OUTPUTS_DIR = REPO_ROOT / "outputs" / f"{NOTEBOOK_NAME}_output"
SCORES_DIR = OUTPUTS_DIR / "scores"
MODEL_PATH = OUTPUTS_DIR / "model.joblib"
RESULTS_DIR = REPO_ROOT / "results" / "classifier" / f"{NOTEBOOK_NAME}_output"
SCORES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_FILES = {
    (dataset, subset): FEATURES_DIR / f"features_{dataset}_{subset}_{SUFFIX}.csv"
    for dataset, subset in [TRAIN, *TEST]
}

files = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "subset": subset,
            "role": "train" if (dataset, subset) == TRAIN else "test",
            "feature_file": path.name,
            "exists": path.is_file(),
        }
        for (dataset, subset), path in FEATURE_FILES.items()
    ]
)
display(files)
print("Features:", FEATURE_COLUMNS)

### Step 2 – Feature tables

**What it does:** Reads the feature tables and counts the recordings of each class (bona fide
and spoof) in every set.

**Why:** To load the previously created feature tables and see what they consist of.

In [ ]:
tables = {key: read_feature_table(path) for key, path in FEATURE_FILES.items()}

rows = []
for (dataset, subset), table in tables.items():
    _, labels = select_features(table)
    rows.append(
        {
            "dataset": dataset,
            "subset": subset,
            "recordings": len(table),
            "usable": len(labels),
            "bonafide": int((labels == "bonafide").sum()),
            "spoof": int((labels == "spoof").sum()),
        }
    )
counts = pd.DataFrame(rows)
display(counts)

### Step 3 – Training

**What it does:** Trains the classifier on all recordings of ASVspoof 2019 LA train. The table
shows the class weights: one bona fide recording counts about 9 times more than a spoofed one,
so both classes have the same total weight.

**Why:** Without the weights the model could learn to answer "spoof" almost every time.

In [ ]:
train_features, train_labels = select_features(tables[TRAIN])
model = train_classifier(train_features, train_labels)

classes = model.classes_
class_weights = pd.DataFrame(
    {
        "label": classes,
        "recordings": train_labels.value_counts().reindex(classes).to_numpy(),
        "weight_per_recording": compute_class_weight("balanced", classes=classes, y=train_labels),
    }
)
class_weights["total_weight"] = class_weights["recordings"] * class_weights["weight_per_recording"]
display(class_weights)

### Step 4 – EER of the test sets

**What it does:** Scores every test set with the model and computes its EER. For some sets also per attack, codec or source.

**Why:** This is the main result: how well the four features work on recordings the model
has not seen.

In [ ]:
scored = {key: add_scores(model, tables[key]) for key in TEST}

eer_table = pd.concat(
    [
        eer_by_group(table, column).assign(dataset=dataset, subset=subset)
        for (dataset, subset), table in scored.items()
        for column in [None, *BREAKDOWNS.get((dataset, subset), [])]
    ],
    ignore_index=True,
)
eer_table = eer_table[["dataset", "subset", "group", "value", "bonafide", "spoof", "eer"]]

for group in ["all", "attack_id", "codec", "source"]:
    print("EER of every test set:" if group == "all" else f"EER per {group}:")
    display(
        eer_table[eer_table["group"] == group].style.format({"eer": "{:.2%}"}).hide(axis="index")
    )

### Step 5 – Scores per codec

**What it does:** For 2021 LA eval and for every source of 2021 DF eval, compares the
recordings after every codec with the recordings without a codec (first row): the typical
(median) score of bona fide recordings without and with the codec and their difference, the
typical score of spoofed recordings with the codec and the typical bandwidth of bona fide
recordings with the codec. All 2021 LA recordings come from ASVspoof 2019 LA eval, so their
source is shown as `asvspoof`.

**Why:** Bona fide recordings should get high scores whatever the codec. A clearly negative
`shift_bonafide` means that the model takes the codec for spoofing – a shortcut. Comparing
inside one source keeps the other differences between the sources out.

In [ ]:
score_table = pd.concat(
    [
        score_medians(part, reference).assign(
            # 2021 LA has no source column: all its recordings come from ASVspoof 2019 LA eval
            dataset=dataset,
            subset=subset,
            source=source or "asvspoof",
        )
        for (dataset, subset), reference in CODEC_REFERENCES.items()
        for source, part in scored[(dataset, subset)].groupby("source")
    ],
    ignore_index=True,
)
score_table = score_table.set_index(["dataset", "subset", "source"]).reset_index()

for (dataset, subset, source), table in score_table.groupby(
    ["dataset", "subset", "source"], sort=False
):
    print(f"{dataset} {subset}, source {source} – scores per codec:")
    display(
        table.style.format(
            {
                "median_score_bonafide": "{:.3f}",
                "median_score_bonafide_with_codec": "{:.3f}",
                "shift_bonafide": "{:+.3f}",
                "median_score_spoof_with_codec": "{:.3f}",
                "median_bandwidth_bonafide_with_codec": "{:.0f}",
            }
        ).hide(axis="index")
    )

### Step 6 – Weights of the features

**What it does:** Shows how much each feature counts in the model and in which direction
(towards spoof or bona fide), next to the typical (median) value of the feature in each class.

**Why:** It shows which shortcut the model relies on most.

In [ ]:
coefficients = feature_weights(model, train_features, train_labels)
display(coefficients.style.format(precision=3).hide(axis="index"))
print(f"Intercept: {model[-1].intercept_[0]:+.3f}")

### Step 7 – Saving the results

**What it does:** Saves the EER table, the scores per codec and the weights to `results/`,
and the model and the score of every recording to `outputs/`.

**Why:** The tables go to git. The model and the scores are needed later, to score the
modified recordings and compare them with these ones.

In [ ]:
SCORE_COLUMNS = ["dataset", "subset", "file_id", "label", "score"]

eer_table.to_csv(RESULTS_DIR / "eer_table.csv", index=False)
score_table.to_csv(RESULTS_DIR / "score_medians.csv", index=False)
coefficients.to_csv(RESULTS_DIR / "coefficients.csv", index=False)
joblib.dump(model, MODEL_PATH)
for (dataset, subset), table in scored.items():
    table[SCORE_COLUMNS].to_csv(SCORES_DIR / f"scores_{dataset}_{subset}.csv", index=False)

for path in [*sorted(RESULTS_DIR.iterdir()), MODEL_PATH, *sorted(SCORES_DIR.iterdir())]:
    print("Saved:", path.relative_to(REPO_ROOT))